In [0]:
catalog = "datafactory"
schema = "incremental_demo"

source_table = f"{catalog}.{schema}.orders_source"
target_table = f"{catalog}.{schema}.orders_target"
control_table = f"{catalog}.{schema}.pipeline_watermark"

pipeline_name = "orders_incremental_v1"

spark.sql("SET TIME ZONE 'UTC'")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {source_table}(
              order_id BIGINT,
              customer_id BIGINT,
              order_status STRING,
              amount DECIMAL(12,2),
              source_updated_at TIMESTAMP,
              ingested_at TIMESTAMP
          )
          USING DELTA
          """)


In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {target_table}(
              order_id BIGINT,
              customer_id BIGINT,
              order_status STRING,
              amount DECIMAL(12,2),
              source_updated_at TIMESTAMP,
              source_ingested_at TIMESTAMP,
              processed_at TIMESTAMP
          )
          USING DELTA
          """)


In [0]:
# each pipeline should have its one row in this table
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {control_table}(
              pipeline_name STRING,
              last_watermark TIMESTAMP,
              updated_at TIMESTAMP
          )
          USING DELTA
          """)

In [0]:
%sql
-- delete from datafactory.incremental_demo.orders_source
-- delete from datafactory.incremental_demo.pipeline_watermark

In [0]:
spark.sql(f"""
          MERGE INTO {control_table} as target
          using (
              select '{pipeline_name}' as pipeline_name,
              cast('1900-01-01 00:00:00' as timestamp) as last_watermark,
              current_timestamp() as updated_at
          ) as source
            ON target.pipeline_name = source.pipeline_name
            WHEN NOT MATCHED THEN 
            INSERT(
                pipeline_name,
                last_watermark,
                updated_at
            ) VALUES(
                source.pipeline_name,
                source.last_watermark,
                source.updated_at
            )
           """)

In [0]:
spark.sql(f"""
          INSERT INTO {source_table}
          values(
              101,
              1001,
              'PLACED',
              120.50,
              TIMESTAMP '2026-08-21 09:00:00',
              TIMESTAMP '2026-08-21 09:01:00'
          ),
          (
              102,
              1002,
              'PLACED',
              250.00,
              TIMESTAMP '2026-08-21 09:05:00',
              TIMESTAMP '2026-08-21 09:06:00'
          ),
          (
              103,
              1003,
              'PLACED',
              400.00,
              TIMESTAMP '2026-08-21 09:10:00',
              TIMESTAMP '2026-08-21 09:11:00'
          )
          """)

In [0]:
display(spark.table(control_table))

In [0]:
display(spark.table(source_table).orderBy("ingested_at"))

Historical load is done, now incremental load follows:

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F, Window

def run_orders_incremental() -> dict:

    #STEP-1 :- Read only the Watermark Row from the Control Table
    #============================================================

    watermark_row = (
        spark.table(control_table)
        .where(F.col("pipeline_name")== pipeline_name)
        .select("last_watermark")
        .first()
    ) 

    if ( 
        watermark_row is None 
        or watermark_row["last_watermark"] is None 
    ):
        raise RuntimeError(
            f"No watermark found for table {pipeline_name}")
    
    last_watermark = watermark_row["last_watermark"]


    #Step-2 :- Capture the Upper Boundary
    #==============================================================
    source_df = spark.table(source_table)

    upper_bound_row = (
        source_df
        .agg(
            F.max("ingested_at").alias(
                "run_upper_bound"
            )
        ).first()
    )
    
    run_upper_bound = upper_bound_row["run_upper_bound"]

    if (
        run_upper_bound is None
        or run_upper_bound <= last_watermark
    ):
        print(
            f"No New Records. ",
            f"Watermark remains {last_watermark}" 
        )
        return {
            "status" : "NO_NEW_DATA",
            "last_watermark" : str(last_watermark)
        }

    #Step-3 :- Read the Bounded Interval
    #==============================================================

    incremental_df = source_df.where(
        (
            F.col("ingested_at") > F.lit(last_watermark)
        ) &
        (
            F.col("ingested_at") <= F.lit(run_upper_bound)
        ) 
    )

    #Step-4 :- Deduplicate Before Merge
    #==============================================================

    latest_change_window = (
        Window
        .partitionBy("order_id")
        .orderBy(
            F.col("source_updated_at").desc_nulls_last(),
            F.col("ingested_at").desc_nulls_last()
        )
    )

    prepared_df = (
        incremental_df
        .withColumn("_row_number",F.row_number().over(latest_change_window))
        .where(F.col("_row_number") == 1)
        .drop("_row_number")
        .withColumn("processed_at",F.current_timestamp())
    )

    # Define when an existing target record should be updated
    update_condition = """
                        t.source_updated_at IS NULL
                        OR s.source_updated_at > t.source_updated_at
                        OR (
                            s.source_updated_at = t.source_updated_at
                            AND (
                                t.source_ingested_at IS NULL 
                                OR 
                                s.ingested_at > t.source_ingested_at
                                )
                            )
                        """

    try:
    #Step-5 :- Insert New Records and Update Existing Records
    #=============================================================

        target = DeltaTable.forName(spark, target_table)

        (
            target.alias("t")
            .merge(prepared_df.alias("s"), "t.order_id = s.order_id")
            .whenMatchedUpdate(
                condition = update_condition,
                set = {
                    "customer_id" : "s.customer_id",
                    "order_status" : "s.order_status",
                    "amount" : "s.amount",
                    "source_updated_at" : "s.source_updated_at",
                    "source_ingested_at" : "s.ingested_at",
                    "processed_at" : "s.processed_at"
                },)
            .whenNotMatchedInsert(
                values = {
                    "order_id" : "s.order_id",
                    "customer_id" : "s.customer_id",
                    "order_status" : "s.order_status",
                    "amount" : "s.amount",
                    "source_updated_at" : "s.source_updated_at",
                    "source_ingested_at" : "s.ingested_at",
                    "processed_at" : "s.processed_at"
                }
            )
            .execute()
        )

        raise RuntimeError(
            "Simulated failure after target MERGE"
        )

    # Prepare the New Watermark
    #=============================================================

        watermark_udpate_df = (
            spark.createDataFrame(
                [
                    (
                        pipeline_name,
                        run_upper_bound
                    )
                ],
                schema = """
                    pipeline_name STRING,
                    last_watermark TIMESTAMP
                    """
            )
            .withColumn(
                "updated_at", F.current_timestamp()
            )
        )

    # Advance the Watermark after Successful Merge
    #=============================================================

        control = DeltaTable.forName(spark, control_table)

        (
            control.alias("t")
            .merge(watermark_udpate_df.alias("s"),
                """
                t.pipeline_name = s.pipeline_name
                """
            )
            .whenMatchedUpdate(
                set={
                    "last_watermark" : "s.last_watermark",
                    "updated_at" : "s.updated_at"
                }
            )
            .whenNotMatchedInsert(
                values={
                    "pipeline_name" : "s.pipeline_name",
                    "last_watermark" : "s.last_watermark",
                    "updat ed_at" : "s.updated_at"
                }
            )
            .execute()
        )

    except Exception:
        print(
            "Pipeline failed. The watermark was "
            "not advanced by this code path, so"
            "the same interval can be replayed."
        ) 
        raise

    print(
        f"Pipeline completed Watermark advanced"
        f"from  {last_watermark} to {run_upper_bound}"
    )

    return{
        "status" : "SUCCESS",
        "previous_watermark" : str(last_watermark),
        "new_watermark" : str(run_upper_bound)
    }

In [0]:
result = run_orders_incremental()

print(result)

In [0]:
display(spark.table(target_table)
        .orderBy("order_id"))

In [0]:
display(
    spark.table(control_table)
    .where(
        f"pipeline_name = '{pipeline_name}'"
    )
)

Now Updates and Inserts

In [0]:
spark.sql(f"""
          INSERT INTO {source_table}
          values(
              101,
              1001,
              'SHIPPED',
              120.50,
              TIMESTAMP '2026-08-21 10:00:00',
              TIMESTAMP '2026-08-21 10:01:00'
          ),
          (
              104,
              1004,
              'PLACED',
              180.75,
              TIMESTAMP '2026-08-21 10:05:00',
              TIMESTAMP '2026-08-21 10:06:00'
          )
          """)

In [0]:
#interval -> 9:11 < ingested_at <= 10:06
result = run_orders_incremental()

print(result)

In [0]:
display(spark.table(target_table)
        .orderBy("order_id")) 

In [0]:
#no new data
result = run_orders_incremental()

print(result)

In [0]:
spark.sql(f"""
          INSERT INTO {source_table}
          values(
              106,
              1006,
              'PLACED',
              140.00,
              TIMESTAMP '2026-08-21 11:05:00',
              TIMESTAMP '2026-08-21 11:06:00'
          )
          """)

In [0]:
#no new data
result = run_orders_incremental()

print(result)

In [0]:
display(spark.table(target_table)
        .orderBy("order_id")) 